# Comparativa de ASR usando BERTScore

Este notebook calcula el BERTScore entre las transcripciones normalizadas (`text_normalized`) y las oraciones de referencia (ground truth).

Se utiliza una versión ligera del modelo para una ejecución rápida, pero se deja comentada la opción de usar un modelo más pesado (transformers grandes) para obtener métricas potencialmente más precisas.

In [1]:
# Instalar dependencias si no están presentes
!pip install bert-score transformers torch pandas numpy protobuf sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import json
from bert_score import score
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## 1. Cargar Datos

In [4]:
# Cargar el dataset normalizado
df = pd.read_csv('/content/drive/MyDrive/asr/dataset_bert.csv')

# Cargar el ground truth
with open('/content/drive/MyDrive/asr/ground_truth.json', 'r') as f:
    ground_truth = json.load(f)

# Crear un diccionario para mapear id -> texto de referencia
ref_dict = {item['id']: item['text'] for item in ground_truth}

# Verificar las primeras filas
df.head()

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...


## 2. Preparar Candidatos y Referencias

In [5]:
# Asegurarse de que la columna 'audio' sea string para hacer el mapeo con el id del json
df['audio'] = df['audio'].astype(str)

# Obtener la lista de candidatos (transcripciones normalizadas)
# Rellenar NaNs con string vacío por si acaso
cands = df['text_normalized'].fillna('').tolist()

# Obtener la lista de referencias correspondientes usando la columna 'audio' (que es el ID)
refs = [ref_dict.get(audio_id, "") for audio_id in df['audio']]

# Verificar que tienen la misma longitud
assert len(cands) == len(refs), "Error: La longitud de candidatos y referencias no coincide."

print(f"Total de pares a evaluar: {len(cands)}")
print(f"Ejemplo candidato: {cands[0]}")
print(f"Ejemplo referencia: {refs[0]}")

Total de pares a evaluar: 6000
Ejemplo candidato: genera 1 cotización para el cliente con fácil con 5 monitores led y 3 soportes de pared
Ejemplo referencia: genera 1 cotización para el cliente compufacil con 5 monitores led y 3 soportes de pared


## 3. Calcular BERTScore

Aquí seleccionamos el modelo. 
*   **Modelo ligero (activo):** `distilbert-base-multilingual-cased` (buen balance velocidad/rendimiento).
*   **Modelo medio (comentado):** `PlanTL-GOB-ES/roberta-base-bne` (mejor para español, más lento que distilbert pero razonable).
*   **Modelo pesado (comentado):** `xlm-roberta-large` (mejor rendimiento, mucho más lento y pesado).

In [7]:
# Calcular BERTScore con MODELO ESTÁNDAR
# Ante los errores persistentes con el modelo PlanTL (KeyError, ValueError, OSError, OverflowError),
# optamos por un modelo estándar multilingüe altamente robusto y soportado nativamente.

# Opción 1: DistilBERT (Ligero y rápido) - Descomentar si se quiere velocidad máxima
# model_type = "distilbert-base-multilingual-cased"

# Opción 2: XLM-RoBERTa Base (Balanceado y robusto) - Recomendado ahora
# model_type = "xlm-roberta-base"

# Opción 3: RoBERTa Large BNE (PlanTL) - Solicitado para GPU/Colab
model_type = "xlm-roberta-large"

print(f"Usando modelo robusto: {model_type} en {device}")

# Eliminamos cualquier parche o configuración extraña. BertScore soporta xlm-roberta nativamente.
# Solo especificamos el modelo y lang="es" para que use las capas correctas por defecto.
P, R, F1 = score(
    cands, 
    refs, 
    verbose=True, 
    model_type=model_type, 
    num_layers=17,
    device=device,
    lang="es",
    rescale_with_baseline=True,  # Mantiene el escalado estadístico
    idf=True
)

# Agregar los resultados al DataFrame
df['bertscore_precision'] = P.cpu().numpy()
df['bertscore_recall'] = R.cpu().numpy()
df['bertscore_f1'] = F1.cpu().numpy()

Usando modelo robusto: xlm-roberta-large en cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

preparing IDF dict...
done in 15.33 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/22 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/94 [00:00<?, ?it/s]

done in 6.29 seconds, 953.33 sentences/sec


## 4. Analizar Resultados

In [8]:
# Mostrar estadísticas descriptivas de los scores
print("Estadísticas de BERTScore F1:")
print(df['bertscore_f1'].describe())

# Mostrar promedio agrupado por proveedor (provider)
if 'provider' in df.columns:
    print("\nPromedio de BERTScore F1 por proveedor:")
    print(df.groupby('provider')['bertscore_f1'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'noise' in df.columns:
    print("\nPromedio de BERTScore F1 por tipo de ruido:")
    print(df.groupby('noise')['bertscore_f1'].mean().sort_values(ascending=False))

# Mostrar promedio agrupado por nivel de ruido (noise)
if 'snr' in df.columns:
    print("\nPromedio de BERTScore F1 por nivel de ruido:")
    print(df.groupby('snr')['bertscore_f1'].mean().sort_values(ascending=False))

Estadísticas de BERTScore F1:
count    6000.000000
mean        0.880456
std         0.189131
min        -0.398195
25%         0.797589
50%         0.999998
75%         1.000000
max         1.000002
Name: bertscore_f1, dtype: float64

Promedio de BERTScore F1 por proveedor:
provider
google    0.918926
custom    0.887422
amazon    0.863708
azure     0.851770
Name: bertscore_f1, dtype: float32

Promedio de BERTScore F1 por tipo de ruido:
noise
clean        0.948457
cafe         0.890291
traffic      0.885510
warehouse    0.842901
Name: bertscore_f1, dtype: float32

Promedio de BERTScore F1 por nivel de ruido:
snr
clean    0.948457
10dB     0.930033
5dB      0.903832
0dB      0.784837
Name: bertscore_f1, dtype: float32


In [13]:
# Guardar el dataframe con los scores si es necesario
# df.to_csv('normalized_dataset_with_bertscore.csv', index=False)
df.to_csv('/content/drive/MyDrive/asr/bert_dataset.csv', index=False)
df.head()

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized,bertscore_precision,bertscore_recall,bertscore_f1
0,p7,1,cafe,0dB,custom,Genera una cotización para el cliente con fáci...,success,1.65,genera 1 cotización para el cliente con fácil ...,0.757242,0.701161,0.729563
1,p7,1,cafe,5dB,custom,Genera una cotización para el cliente con PUC ...,success,1.56,genera 1 cotización para el cliente con puc fa...,0.659615,0.700823,0.680736
2,p7,1,cafe,10dB,custom,Genera una cotización para el cliente con Puff...,success,1.59,genera 1 cotización para el cliente con puffaz...,0.438205,0.703294,0.568640
3,p7,1,clean,clean,custom,Genera una cotización para el cliente con FooF...,success,1.66,genera 1 cotización para el cliente con foofac...,0.450867,0.725140,0.585667
4,p7,1,traffic,0dB,custom,"genera una cotización para el cliente fácil, c...",success,1.53,genera 1 cotización para el cliente fácil con ...,0.232345,0.369568,0.301425


In [15]:
# Cargar bert_dataset.csv y mostrar los 20 registros con menor bertscore_f1
import pandas as pd
df_bert = pd.read_csv('/content/drive/MyDrive/asr/bert_dataset.csv')
peores_20 = df_bert.nsmallest(5, 'bertscore_f1')
display(peores_20)

,person,audio,noise,snr,provider,text,status,transcription_time,text_normalized,bertscore_precision,bertscore_recall,bertscore_f1
4467,p5,12,warehouse,0dB,google,3 1 15 3 7 0 9 22 0 7 83 66 0 0 1,success,1.18,3115370922078366001,-0.277603,-0.518093,-0.398195
3714,p10,12,traffic,0dB,google,09 22 07 83 66 001,success,0.93,0922078366001,-0.119167,-0.630784,-0.385426
3637,p10,4,warehouse,0dB,google,Quiero una persona para mi mejor.,success,0.89,quiero 1 persona para mi mejor,-0.142415,-0.373241,-0.258131
5437,p7,4,warehouse,0dB,google,una plataforma para mi motor,success,0.85,1 plataforma para mi motor,-0.103074,-0.283938,-0.192910
1440,p2,10,cafe,0dB,custom,de la marca Dura Vista.,success,1.09,de la marca dura vista,-0.162334,-0.220566,-0.189449


In [16]:
# Tabla: promedio de METEOR por proveedor y nivel de ruido
tabla_bert = df_bert.pivot_table(
    values='bertscore_f1',
    index='provider',
    columns='snr',
    aggfunc='mean'
).round(4)
display(tabla_bert)

snr,0dB,10dB,5dB,clean
provider,,,,
amazon,0.7948,0.8986,0.8851,0.9018
azure,0.7429,0.9094,0.8743,0.9380
custom,0.7787,0.9439,0.9127,0.9681
google,0.8229,0.9682,0.9433,0.9859
